**Table of contents**<a id='toc0_'></a>    
1. [Pre-requesite](#toc1_)    
1.1. [Configure project and data paths](#toc1_1_)    
1.2. [Initialise the DataPreprocessor](#toc1_2_)    
2. [Generate the file metadata](#toc2_)    
3. [Standardise file name and column name](#toc3_)    
4. [(Optional) Pre-cleaning validation](#toc4_)    
5. [Data cleaning](#toc5_)    
6. [(Optional) Post-cleaning validation](#toc6_)    
7. [(Optional) Process column metadata](#toc7_)    
7.1. [Appending descriptions and values to column metadata](#toc7_1_)    
7.2. [Curate data types recorded in column metadata](#toc7_2_)    
8. [(Optional) Generate data profile](#toc8_)    
9. [Data merging](#toc9_)    
10. [Data cleaning: remove duplicate data ](#toc10_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="background-color:#e8f4f8; border: 2px solid #3399cc; padding: 15px; border-radius: 8px;">

<h1><b>Step-by-step Guidance - Running the NEETML Tool</b></h1>

<!-- **Author:** Yanhua Xu -->

<!-- <div style="margin: 2px 0 10px 0;">
  <span style="display: inline-block; font-size: 20px; color: #2f4f5f; background-color: rgba(255,255,255,0.55); padding: 5px 10px; border-radius: 999px;">
    <span style="letter-spacing: 0.4px; text-transform: uppercase; font-size: 20px; font-weight: 700; color: #3399cc;">Author</span>
    <span style="color:#9aa; margin: 0 7px;">•</span>
    <span style="font-weight: 600;">Yanhua Xu</span>
  </span>
</div> -->

<span style="display: inline-block; font-size: 20px; color: #2f4f5f; background-color: rgba(255,255,255,0.55); padding: 5px 10px; border-radius: 999px; margin: 2px 0 16px 0;">
  <span style="letter-spacing: 0.4px; text-transform: uppercase; font-size: 20px; font-weight: 700; color: #3399cc;">Author</span>
  <span style="color:#9aa; margin: 0 7px;">•</span>
  <span style="font-weight: 600;">Yanhua Xu</span>
</span>

<!-- <div style="background-color:#e8f4f8; border: 2px solid #3399cc; padding: 15px; border-radius: 8px;"> -->

<div style="font-size: 40px; font-weight: 500; color: #444; border-top: 1px solid #999; padding-top: 8px; margin-top: 6px;">
Step 1 - Raw Data Preparation and Cleaning
</div>

<!-- # **Step 1 - Raw Data Preparation and Cleaning** -->
<!-- <h><b> Step 1 - Raw Data Preparation and Cleaning </b></h1> -->

This notebook prepares and cleans the raw datasets to ensure structural and semantic consistency before any feature-level transformation.

It includes:  
- Generating and validating file metadata  
- Standardising file and column names  
- Performing data-quality and validation checks  
- Generating data profiles (optional)
- Cleaning and deduplicating records  
- Merging and aggregating data from multiple sources  

⚠️ **Important:**  
 - This step **does not introduce new variables** or modify existing data values.  
 - The goal is to produce a harmonised, analysis-ready dataset while fully preserving the original data structure.  
 - All operations that **add new information or change data values** (e.g., integrating external datasets such as IMD or School Performance data) are performed in **Step 2**.
 - Some steps may take longer time

</div>

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">

**🚀 Future Change**

- solve path issue, add path of neetml to sys.path to make sure this script can be run successfully
- enable the function to extract column metadata from folder
- introduce a lazy, user-configurable path layer (`neetml.paths`).
- download necessay public dataset automatically

</blockquote>

# 1. <a id='toc1_'></a>[Pre-requesite](#toc0_)

## 1.1. <a id='toc1_1_'></a>[Configure project and data paths](#toc0_)

NEETML uses one project-wide settings layer. 

The resolution order is **explicit function argument > environment variable >
`.neetml/config.yaml` > repository default**. 

Relative paths are resolved from the configured **project root** rather than the current notebook directory.

<details>
<summary>🚀 Temporary override (current session only)</summary>

```python
%env NEETML_PROJECT_ROOT="/path/to/your/project"
```

or

```bash
export NEETML_PROJECT_ROOT="/path/to/your/project"
```

</details>

<details>
<summary>⚙️ Save a persistent configuration</summary>

```python
settings.update(
    project_root="/path/to/your/project",
    data_dir="/path/to/project-data",
    dataset="model_input",
    folder="/secure/neet",
    year_range=(2018, 2026),
).save()
```

</details>

<details>
<summary>📌 Additional notes</summary>

- `data_dir` moves the **raw**, **interim**, and **processed** directories together.
- `.neetml/config.yaml` is ignored by Git.
- Reload `NEETMLConfig` after saving before continuing.

</details>

In [1]:
%env NEETML_LOG_LEVEL=DEBUG # optional

env: NEETML_LOG_LEVEL=DEBUG # optional


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

try:
    import neetml
except:
    # Source-checkout bootstrap only; installed packages do not need this block.
    bootstrap_root = next(
        (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
         if (candidate / "neetml").is_dir()),
        None,
    )
    if bootstrap_root is not None and str(bootstrap_root) not in sys.path:
        sys.path.insert(0, str(bootstrap_root))

from neetml.config import NEETMLConfig

# repository defaults
settings = NEETMLConfig.load()
settings.get_path(keys="all")

,Type,Path,Description
0,project_root,/Users/alysia/Research-NEET,Configured project root
1,user_config,/Users/alysia/Research-NEET/.neetml/config.yaml,User configuration
2,data_dir,/Users/alysia/Research-NEET/data,Configured path
3,raw_dir,/Users/alysia/Research-NEET/data/00_source,Configured path
4,raw_meta_dir,/Users/alysia/Research-NEET/data/00_source_meta,Configured path
5,external_dir,/Users/alysia/Research-NEET/data/00_external,Configured path
6,interim_dir,/Users/alysia/Research-NEET/data/01_source_colstd,Configured path
7,processed_dir,/Users/alysia/Research-NEET/data/02_processed,Configured path
8,file_meta_path,/Users/alysia/Research-NEET/data/02_processed/...,Configured path
9,col_meta_path,/Users/alysia/Research-NEET/data/02_processed/...,Configured path


## 1.2. <a id='toc1_2_'></a>[Initialise the DataPreprocessor](#toc0_)

Before proceeding, make sure to initialise the `DataPreprocessor`.  

You can either use **custom path settings** or **default paths**.

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Click to view example using custom path settings
</summary>

```python
from pathlib import Path

# Base directory for data
BASE_DATA_DIR = Path('../data')

# Define subdirectories
SRC_DATA_DIR = BASE_DATA_DIR / '00. source'
SRC_META_DIR = BASE_DATA_DIR / '00. source_metadata'
EXT_DATA_DIR = BASE_DATA_DIR / '00. external'
SRC_COLSTD_DIR = BASE_DATA_DIR / '01. source_colstd'              # raw data with standardised column names
PROC_DATA_DIR = BASE_DATA_DIR / '02. processed'

# Ensure directories exist
for path in [SRC_DATA_DIR, SRC_META_DIR, SRC_COLSTD_DIR, PROC_DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Define metadata file paths
FILE_METADATA_PATH = PROC_DATA_DIR / 'file_metadata.xlsx'
COL_METADATA_PATH = PROC_DATA_DIR / 'col_metadata.xlsx'

# Define configuration dictionary
configs = {
    'src_colstd_dir': SRC_COLSTD_DIR,
    'file_metadata_path': FILE_METADATA_PATH,
    'col_metadata_path': COL_METADATA_PATH,
    'src_data_dir': SRC_DATA_DIR,
    'src_meta_dir': SRC_META_DIR,
    'ext_data_dir': EXT_DATA_DIR,
    'proc_data_dir': PROC_DATA_DIR,
    'clean_dir': '1. cleaned',
    'merge_dir': '2. merged',
}

# Initialise the DataPreprocessor with custom paths
preprocessor = DataPreprocessor(**configs)
```

</details>

In this notebook, we demonstrate the use of default path settings,

so you can simply initialise the `DataPreprocessor` as follows:

In [3]:
from neetml.preprocessing import DataPreprocessor

preprocessor = DataPreprocessor(settings=settings)

You can check the default paths by calling `get_path`:

In [4]:
preprocessor.get_path(keys='all')

,Type,Path,Description
0,data_config,/Users/alysia/Research-NEET/neetml/configs/dat...,YAML file for data manifest and configuration
1,file_metadata,/Users/alysia/Research-NEET/data/02_processed/...,Excel file containing file-level metadata
2,col_metadata,/Users/alysia/Research-NEET/data/02_processed/...,Excel file containing column-level metadata
3,source,/Users/alysia/Research-NEET/data/00_source,Folder where source downloaded data are stored
4,source_metadata,/Users/alysia/Research-NEET/data/00_source_meta,Folder for source metadata files
5,source_colstd,/Users/alysia/Research-NEET/data/01_source_colstd,Folder for raw data with standardised column n...
6,clean,/Users/alysia/Research-NEET/data/02_processed/...,Folder for cleaned and validated datasets
7,merged,/Users/alysia/Research-NEET/data/02_processed/...,Folder for merged datasets after linkage
8,profiles,/Users/alysia/Research-NEET/data/02_processed/...,Folder for profiled or derived data products


# 2. <a id='toc2_'></a>[Generate the file metadata](#toc0_)

Getting metadata for files is essential for our data processing workflow (for now) because we have many files and tables with inconsistent naming conventions. This inconsistency poses a challenge for future users, especially those from cities outside of Bradford Council, as their naming conventions might not align with Bradford Council. 

Therefore, we are preparing for the standardisation of file names. Part of this process includes a step called `Auto Curation`. `Auto Curation` helps address the variability in file naming that might occur when different individuals name files. It replies on the Data Configuration specified in the YAML file to generate consistent Data Categories, Y11 Cohorts, and Yearly Data Information.
However, some files may still require manual information supplementation because they do not fit into the defined rules.


<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Example of File Metadata Format (click to expand to)
</summary>

<!-- **Example of File Metadata Format** -->

If users wish to provide their own file metadata, it must follow the format below:

| Category  | File Name                                                  | Cohort Y11 AY | Sheet Name             | Year Group       | Year Group AY          |
|----------------|------------------------------------------------------------|------------|------------------------|------------------|-----------------|
| suspPermExcl   | Exclusions and Suspensions              | 2024-2025  | Y10 in Y7 (2020 - 2021)    | 7              | Y7 2020-2021    |

***Note:***: If the `standardise_fnames_colnames()` function is used to generate file metadata automatically, additional columns such as **Needs Review** and **Standard File Name** will be generated. These columns are removed in the user-provided format shown above but are included in the automatically generated file metadata to assist in the standardisation process.

</details>

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Column Definitions (click to expand to)
</summary>

<!-- **Column Definitions** -->

- **Category**: The category of data, such as `suspPermExcl` (for all perm exclusion and suspension (fixed-term exclusion) data), `attendance`, `post16Dest`, `sepGuarantee`, `ks2`, `ks4` and `census`.
- **File Name**: The original name of the file.
- **Cohort Y11 AY**: Academic year when cohort reached Y11
- **Sheet Name**: The specific worksheet name within the file, used to accurately locate the data source.
- **Year Group**: The year group within the cohort, where 7.0 represents Year 7.
- **Year Group AY**: The year group and academic year when the cohort entry into this year group.

</details>


In [5]:
# Generate the file info
file_metadata, _ = preprocessor.extract_file_metadata()


+-------------------------------------+
|  Starting extracting file metadata  |
+-------------------------------------+

2026-07-22 23:11 - INFO - Number of files found in /Users/alysia/Research-NEET/data/00_source: 55
2026-07-22 23:11 - WARNING - Rows that need review:
╒══════════════╤═══════════════════════════════════════════════════════════╤═════════════════╤═════════════════════════════════╤══════════════╤═════════════════╤══════════════════╤═════════════════════╤═════════════╤════════════════╤══════════╤═════════════════════════════════╕
│ Category     │ File Name                                                 │ Cohort Y11 AY   │ Sheet Name                      │   Year Group │ Year Group AY   │   Raw Year Group │ Raw Year Group AY   │   Row Count │ Needs Review   │ Notes    │ Standard File Name              │
╞══════════════╪═══════════════════════════════════════════════════════════╪═════════════════╪═════════════════════════════════╪══════════════╪═════════════════╪═════════

Now have a look at the first 5 rows of the file information we generated:

In [6]:
file_metadata.tail()

,Category,File Name,Cohort Y11 AY,Sheet Name,Year Group,Year Group AY,Raw Year Group,Raw Year Group AY,Row Count,Needs Review,Notes,Standard File Name
144,suspPermExcl,2024 Y10 Exclusions and Suspensions ANON (Y11 ...,2024 - 2025,Y10 in Y7 (2020_21),7.0,Y7 2020 - 2021,7.0,7 2020 - 2021,4025.0,,NaN,2024-2025_suspPermExcl_Y7.xlsx
145,suspPermExcl,2024 Y10 Exclusions and Suspensions ANON (Y11 ...,2024 - 2025,Y10 in Y8 (2021_22),8.0,Y8 2021 - 2022,8.0,8 2021 - 2022,4025.0,,NaN,2024-2025_suspPermExcl_Y8.xlsx
146,suspPermExcl,2024 Y10 Exclusions and Suspensions ANON (Y11 ...,2024 - 2025,Y10 in Y9 (2022_23),9.0,Y9 2022 - 2023,9.0,9 2022 - 2023,4025.0,,NaN,2024-2025_suspPermExcl_Y9.xlsx
147,suspPermExcl,2024 Y9 Exclusions and Suspensions ANON (Y11 2...,2025 - 2026,Y9 in Y7 (2021_22),7.0,Y7 2021 - 2022,7.0,7 2021 - 2022,3935.0,,NaN,2025-2026_suspPermExcl_Y7.xlsx
148,suspPermExcl,2024 Y9 Exclusions and Suspensions ANON (Y11 2...,2025 - 2026,Y9 in Y8 (2022_23),8.0,Y8 2022 - 2023,8.0,8 2022 - 2023,3935.0,,NaN,2025-2026_suspPermExcl_Y8.xlsx


# 3. <a id='toc3_'></a>[Standardise file name and column name](#toc0_)

<blockquote style="background-color: #FFFFE0; padding: 15px; margin: 0; border-left: 5px solid #FFD700;">
- In the previous step, the file metadata was already generated. However, if the previous step was not executed, the file metadata will be automatically generated in this step, or the user can provide their own file metadata by setting <code>preprocessor.stndardise_fnames_colnames(file_metadata_path=the path of your file metadata)</code>.
- The column metadata will be generated after this step.
</blockquote>


<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Standardised File Name (click to expand to)
</summary>
<!-- **Standardised File Name** -->

Each file will be renamed based on its corresponding data category, Cohort Y11 AY, and optionally Year Group AY or Year Group, as specified in the file metadata Excel file stored in `file_metadata_PATH`.

The standardised filenames are dynamically built using the `file_naming_format`, which defaults to:  
`["cohort_y11_ay", "category", "year_group"]`.

The standardised filenames must follow one of these formats:
1. Default format:
   `{cohort_y11_ay}_{category}_{year_group}.xlsx`  
   *(e.g., `2023-2024_attendance_Y11.xlsx`)*

2. If `file_naming_format` contains `cohort_yg_ay` instead of `year_group`:
   `{cohort_y11_ay}_{category}_{cohort_yg_ay}.xlsx`  
   *(e.g., `2023-2024_attendance_Y11-2019-2020.xlsx`)*

3. If both `cohort_yg_ay` and `year_group` are in `file_naming_format` (4 components):
   - Only the component that appears **first** in the `file_naming_format` will be retained.  
   *(e.g., If `["category", "cohort_y11_ay", "cohort_yg_ay", "year_group"]`, the `cohort_yg_ay` is prioritised: `attendance_2023-2024_Y11-2019-2020.xlsx`)*

**Examples:**
- NCCIS data: 
  `{category}-{cohort_y11_ay or recorded month and year}.xlsx`  
  *(e.g., `post16Dest_2024-Mar.xlsx`)*

- General data:
  `{category}_{cohort_y11_ay}_in_{year_group}-{cohort_yg_ay}.xlsx`  
  *(e.g., `activitySurvey_2018_in_Y11-Y8-2015-2016.xlsx`)*  

</details>

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Standardised Column Name (Snake Case) (click to expand to)
</summary>

<!-- **Standardised Column Name (Snake Case)** -->

Each column will be renamed based on the `standardise_colnames_rule` in the data config file stored in `CONFIG_PATH`. The rules are as follows:

- Pre-defined Replacement Steps

| **Step** | **Original Value**            | **Replacement**         | **Description**                                    |
|----------|-------------------------------|-------------------------|----------------------------------------------------|
| 1        | `' '` (space)                  | `'_'`                   | Replace all spaces with underscores                |
| 2        | `'-'` (dash)                   | `'_'`                   | Replace all dashes with underscores                |
| 3        | `'Student_ID'`                 | `'Stud_ID'`             | Replace `Student_ID` to `Stud_ID`                   |
| 4        | `'StudID'`                     | `'Stud_ID'`             | Replace `StudID` to `Stud_ID`                       |
| 5        | `'DateofConfirmation'`         | `'Date_of_Confirmation'`| Replace `DateofConfirmation` to `Date_of_Confirmation`|
| 6        | `'SENprovision'`               | `'SEN_Provision'`       | Replace `SENprovision` to `SEN_Provision`           |
| 7        | `'SENSupport'`                 | `'SEN_Support'`         | Replace `SENSupport` to `SEN_Support`               |
| 8        | `'NCCISCohort'`                | `'NCCIS_Cohort'`        | Replace `NCCISCohort` to `NCCIS_Cohort`             |
| 9        | `'NCyearActual'`               | `'NC_Year_Actual'`      | Replace `NCyearActual` to `NC_Year_Actual`          |
| 10       | `'NCyear'`                     | `'NC_Year'`             | Replace `NCyear` to `NC_Year`                       |
| 11       | `'SENneed'`                    | `'SEN_Need'`            | Replace `SENneed` to `SEN_Need`                     |
| 12       | `'noSpeccon'`                  | `'no_spec_con'`         | Replace `noSpeccon` to `no_spec_con`                |
| 13       | `'SENunitIndicator'`           | `'SEN_Unit_Indicator'`  | Replace `SENunitIndicator` to `SEN_Unit_Indicator`  |

- Other Formatting Operations

| **Step** | **Operation**                  | **Description**                                    |
|----------|---------------------------------|----------------------------------------------------|
| 12       | Add underscore before caps      | Add `_` before each capital letter (e.g., `ActivityCode` becomes `Activity_Code`) |
| 13       | Convert to lowercase            | Convert all column names to lowercase (e.g., `Stud_ID` becomes `stud_id`) |
| 14       | Remove underscore around '*' and '%' | Remove underscore around '*' and '%' (e.g., `_%` becomes `%`) |

- Optional Formatting Operations

| **Step** | **Operation**     | **Description**                                      |
|----------|------------------|------------------------------------------------------|
| 15       | Add prefix       | Add file's data category as a column prefix (e.g., `school` in attendance data becomes `attendance_school`) |


***Note:*** 
- If words are not separated by spaces or capital letters, or if there's inconsistent capitalisation (e.g., one word uses uppercase and another does not, like `SENprovision`), underscores will not be automatically added. These cases require manual handling by expanding the `standardise_rules` in `data_config.yaml` to define the correct formatting. Mixed-case column names need handling because all column names are converted to lowercase for consistency. To improve readability, underscores are recommended to be added between words to clarify column names. A <span style="color:red;">warning</span> will be raised for mixed-case column names to prompt a review.

</details>

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Handling Column Names (click to expand to)
</summary>

**Handling Columns with Duplicate Names:**
The source table may contains columns with same name and we will only keep the first non-empty one:
- Drop completely empty duplicate columns.
- Keep the **first** non-empty column if multiple duplicates exist.
- Retain one column if all duplicates are empty.

**Col Metadata**

The column metadata will be generated and saved in an Excel file. If the metadata file already exists, it will be updated with any new columns, and a backup of the existing data will be created to preserve previous metadata. The metadata is saved as separate sheets for each data category.

Each column's metadata includes the following information:

| **Field**           | **Description**                                                                 |
|--------------------|---------------------------------------------------------------------------------|
| Column Name      | The standardised column name.                                                  |
| Source Column    | The original column name from the dataset.                                     |
| Data Type        | The primary detected data type.                                                |
| Detected Data Types | All detected data types (if multiple were found).                              |
| Num Unique Values| The number of unique values in the column.                                     |
| Unique Values    | A list of unique values in the column.                                         |
| Filenames        | Standardised filenames where the column appears.                               |
| Source Filenames | Original filenames where the column appears.                                   |

</details>



In [7]:
# Process all files listed in the directory
# will take a few minutes
preprocessor.standardise_fnames_colnames(
  add_prefix=True,
  overwrite=False
)


+-----------------------------------------------+
|  Standardising filenames and column names...  |
+-----------------------------------------------+

2026-07-22 23:11 - INFO - /Users/alysia/Research-NEET/data/02_processed/file_metadata.xlsx loaded.
File standardisation process completed.


# 4. <a id='toc4_'></a>[(Optional) Pre-cleaning validation](#toc0_)

In [8]:
# # will take a few minutes
# preprocessor.validate_files_and_colnames(
#   folder_path=RAW_DATA_PATH,
#   validate_dup_file=False,
#   common_cols_threshold=0.6
# )

# 5. <a id='toc5_'></a>[Data cleaning](#toc0_)

In this step, we clean each table based on the following rules:

1. **Remove rows with missing `stud_id` (Student ID):**
   - Rows where the `stud_id` column contains NaN values are removed.
   - This is controlled by the parameter `rm_nan_stud_id=True`.

2. **Remove empty columns:**
   - Columns where all values are NaN are removed from the table.
   - This is controlled by the parameter `rm_empty_cols=True`.

3. **Remove columns with only one unique value (constant columns):**
   - Columns where all values are identical are removed from the table.
   - This is controlled by the parameter `rm_constant_cols`, which can be set to `global`, `local` or `False`.
   - If the parameter `rm_constant_cols` is set to `global`, columns with constant values across all files in the folder are removed.
     - Globally Constant Columns are defined as columns that have a single dominant constant value in at least a specified proportion of files. This proportion is controlled by the `constant_consistency` parameter (e.g., `0.8` for 80%). The default value for this parameter is `0.8`. Columns qualify as globally constant if:
        - They have a single unique non-missing value in the specified proportion of files.
        - The column is mostly empty in other files or has minimal variation.
   - If the parameter `rm_constant_cols` is set to `local`, columns with constant values within the current file are removed.

4. **Remove duplicates:**
   - If duplicate rows are found, the first instance is kept, and the rest are removed (`rm_dups_threshold = 'first'`).
   - Alternatively, duplicates can be removed based on the percentage of NaN (i.e., missing) values in duplicate rows (e.g., `rm_dups_threshold = 0.3`).

5. **(Optional) Remove problematic Student IDs:**
   - Rows containing specific problematic Student IDs (e.g., `[471807, 534073]`) are removed.
   - This is controlled by the parameter `rm_problematic_ids`.

6. **(Optional) Remove columns with more than a specified percentage of missing values:**
   - Columns where the percentage of missing values exceeds the specified threshold are removed.
   - This is controlled by the parameter `rm_nan_cols_threshold`, which can be set to a float value (e.g., `0.95` for 95%).
   - If the parameter `rm_nan_cols_threshold` is set to `False`, no columns will be removed based on the percentage of missing values.

7. **Remove sensitive columns:**
   - Columns containing sensitive data, specified by column name, are removed if listed in `rm_sensitive_cols`.

***Note:***
- Rows where only the `stud_id` is present but other fields are missing are not removed unless they meet the specified conditions listed above.

----

**Other Operations:**
- If a student has multiple entries (duplicate Student IDs detected), those records will be printed out. For better readability, only the first two students with such cases will be shown.

In [9]:
# _ = preprocessor.identify_globally_constant_columns(
#   RAW_DATA_PATH,
#   missing_cutoff=0.95,
# )

In [10]:
preprocessor.clean_data(
    rm_nan_stud_id=True,
    rm_nan_cols_threshold=False,
    rm_empty_cols=True,
    rm_constant_cols='global', # need few mins
    rm_dups_threshold='first',
    # rm_problematic_ids=[471807, 534073],
    consider_missing=False,
    missing_cutoff=0.95,
    rm_sensitive_cols=['surname', 'surname', 'middle_names', 'client_no', 'ukprn', 'ypid',
                       'upn', 'uln', 'pupil_matching_ref', 'candno', 'cand_id', 'ks4_pupilmatchingref', 'rec_no'],
    overwrite=False,
)


+-----------------------------------------+
|  Starting the data cleaning process...  |
+-----------------------------------------+

Data cleaning process completed.


# 6. <a id='toc6_'></a>[(Optional) Post-cleaning validation](#toc0_)

> **Note:** Feature drift may occur in *attendance*, *KS2*, and *exclusion* data, as both the number of features and their names have been observed to change over time.

In [11]:
# preprocessor.validate_files_and_colnames(
#   folder_path=PREP_DATA_PATH / "1. cleaned_data",
#   validate_dup_file=False,
#   common_cols_threshold=0.60
#   )

# 7. <a id='toc7_'></a>[(Optional) Process column metadata](#toc0_)

## 7.1. <a id='toc7_1_'></a>[Appending descriptions and values to column metadata](#toc0_)

The column metadata generated in previous steps includes Python-recognised data types. However, this may not always be fully accurate, as numerical labels might be used in datasets. If additional metadata files containing variable descriptions and value mappings are available, this step integrates them into the column metadata generated earlier.

This step is optional, as the main goal is to make it easier to sort out data types. It consolidates all the descriptions and value mappings into a single file, so you don't have to keep referring to multiple metadata files. You can simply rely on this one file to get the job done.

**What You Need to Provide**

1. **Column Metadata**  

   A Pandas DataFrame or Excel sheet for each data category, generated in earlier steps. It must include either the `Column Name` or `Source Column`.  

   **Example Format**:  
   
   <!-- | Column Name      | Data Type | Detected Data Types |
   |------------------|-----------|---------------------|
   | attendance_rate  | float64   | float64             |
   | absence_days     | int64     | int64               | -->

   <table>
      <thead>
         <tr>
            <th>Column Name</th>
            <th>Data Type</th>
            <th>Detected Data Types</th>
         </tr>
      </thead>
      <tbody>
         <tr>
            <td>attendance_rate</td>
            <td>float64</td>
            <td>float64</td>
         </tr>
         <tr>
            <td>absence_days</td>
            <td>int64</td>
            <td>int64</td>
         </tr>
      </tbody>
   </table>

   &nbsp;

2. **YAML Metadata Configuration**  

   This specifies how to append descriptions and value mappings from metadata files.  

   - If there's more than one source, `nicknames` help organise and distinguish the appended information.  
   - Each source's nickname is defined in the YAML configuration.  
   - If no nickname is provided, the default columns named `Description` and `Values` are used.  
   - If conflicts arise (e.g., the same variable has different descriptions from multiple sources), new columns are created with the nicknames as suffixes to resolve the differences.
  
   **Example YAML Format**:  

   ```yaml
   attendance:
     metadata:
       sources:
         - filename: metadata.xlsx
           sheetname: Attendance
           colname: Field
           description: Info
           value: Values
           nickname: Source1
         - filename: additional_metadata.xlsx
           sheetname: ExtraInfo
           colname: Variable
           description: Details
           value: Mappings
           nickname: Source2
  ```

In [12]:
col_metadata = preprocessor.add_col_info()

2026-07-22 23:11 - INFO - No changes detected. File not updated.


## 7.2. <a id='toc7_2_'></a>[Curate data types recorded in column metadata](#toc0_)

Before running the next steps, we need to make sure that each column has a clear and consistent data type recorded in the metadata.

This step reviews and standardises the data types recorded in the column metadata. Before this step, some columns may have data types recorded in different formats, even when they describe the same kind of data. For example, dates, numbers, yes/no values, and categories may not always be labelled consistently.

After this step, each column will have a clearer and more consistent data type recorded in the metadata. This makes the metadata easier to review and helps later steps understand how each column should be treated.

**What happens in this step**

In this step:

- the original recorded data type is kept for reference;
- a curated data type is added or updated;
- common data type labels are standardised;
- possible yes/no fields are identified as `boolean`, where appropriate;
- changes between the original and curated data types are highlighted for review.

**Examples of changes**

For example, this step may make changes like:

| Before curation | After curation |
|---|---|
| `timestamp` | `datetime` |
| `int64` | `Int64` |
| `double` | `Float64` |
| `bool` | `boolean` |
| yes/no values such as `0 = no, 1 = yes` | `boolean` |

**What does not change**

This step only changes the **column metadata**.

It does **not**:

- change the raw data values;
- add new variables;
- remove any columns from the dataset.

**Output and review**

The main output is an updated metadata file. A highlighted version is also produced so that any changed data type decisions can be checked before moving on.

After running this step, open the highlighted Excel file. Cells highlighted in red show data types that were changed during curation.

Review these highlighted changes before using the cleaned metadata file in the next step.

In [13]:
col_metadata = preprocessor.curate_col_dtype()

2026-07-22 23:11 - INFO - No changes detected. File not updated.


# 8. <a id='toc8_'></a>[(Optional) Generate data profile](#toc0_)

This step is optional, used for creating data profiling reports for the cleaned datasets.

A data profiling report gives a quick overview of what is inside each dataset. It helps us check the data before moving on to the next steps.

The report can show information such as:

- how many rows and columns the dataset has;
- what data type each column has;
- how many missing values there are;
- how many unique values each column contains;
- basic summaries for numeric and categorical columns;
- possible patterns or issues in the data.

This step can create a report for either:

- one dataset loaded as a DataFrame; or
- all `.csv` and `.xlsx` files in a selected folder.

If a curated data schema is provided, the schema is applied before profiling so that the columns are interpreted using the corrected data types.

The main output is an HTML profiling report.

After running this step, open the generated report and check whether the dataset looks as expected. Pay particular attention to missing values, unexpected data types, unusual values, and columns with very few or too many unique values.

In [14]:
# preprocessor.create_data_profile()

# 9. <a id='toc9_'></a>[Data merging](#toc0_)

The purpose of this step is to merge data from different sources to create a complete picture of each student without changing the original values.

The following two tables illustrate different schedules for merging School and NEET-related data.

<style>
    table {
        width: 100%;
        table-layout: fixed;
        word-wrap: break-word;
        white-space: normal;
        border-collapse: collapse;
    }
    th, td {
        padding: 8px;
        text-align: left;
        border: 1px solid #ddd;
    }
    /* Light Green for Year Group and Academic Age */
    th.group-info {
        background-color: #d4edda;
        color: black;
    }
    /* Light Yellow for Data Sources */
    th.group-data {
        background-color: #fff3cd;
        color: black;
    }
    /* Light Red for Phase One Prediction */
    th.phase-one {
        background-color: #ffcccb;
        color: black;
    }
    /* Light Blue for Phase Two Prediction */
    th.phase-two {
        background-color: #add8e6;
        color: black;
    }
    .prediction-note {
        margin-top: 10px;
        padding: 10px;
        background-color: #f8f9fa;
        border-left: 4px solid #6c757d;
    }
</style>

1. Data Schedule Based on Actual Availability
    Scheduled according to when the data is realistically collected and available.
    In this setup, KS4 and NCCIS data are appended from Year 12 onwards, as they are only available after the end of Year 11.

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Table 1 — Data Schedule Based on Actual Availability (click to expand)
</summary>
<table>
    <tr>
        <th class="group-info">Year Group</th>
        <th class="group-info">Academic Age</th>
        <th class="group-data">Sep / Oct (Autumn Term) Data Sources</th>
        <th class="phase-one">Phase 1: Predict Pre-Spring Outcome</th>
        <th class="group-data">July (Summer Term) Data Sources</th>
        <th class="phase-two">Phase 2: Predict Post-Spring Outcome</th>
    </tr>
    <tr>
        <td>Y7</td>
        <td>11</td>
        <td></td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y7 School Data</td>
        <td></td>
    </tr>
    <tr>
        <td>Y8-Y10</td>
        <td>12-14</td>
        <td></td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + School Data for the Corresponding Year Group</td>
        <td></td>
    </tr>
    <tr>
        <td>Y11</td>
        <td>15</td>
        <td></td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y11 School Data</td>
        <td></td>
    </tr>
    <tr>
        <td>Y12</td>
        <td>16</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 16)</td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 16)</td>
        <td>NCCIS September Extract in the Following Year (Age 17)</td>
    </tr>
    <tr>
        <td>Y13</td>
        <td>17</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 17)</td>
        <td>NCCIS March Extract (Age 17)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 17)</td>
        <td>NCCIS September Extract in the Following Year (Age 18)</td>
    </tr>
    <tr>
        <td>Y14+</td>
        <td>18-24</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 18+)</td>
        <td>NCCIS March Extract (Age 18+)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 18+)</td>
        <td>NCCIS September Extract in the Following Year (Age 19+)</td>
    </tr>
</table>
<div class="prediction-note">
    <strong>Prediction outcome timing:</strong><br>
    <strong>Phase 1 — Pre-Spring Outcome</strong> refers to the student's outcome recorded in the March NCCIS extract.<br>
    <strong>Phase 2 — Post-Spring Outcome</strong> refers to the student's outcome recorded in the September NCCIS extract in the following academic year.
</div>
</details>

2. Data Schedule Based on Earliest Applicable Usage
    Scheduled according to the earliest reasonable time the data could be used.
    In this setup, KS4 and NCCIS data are appended starting from Year 11, since the information is typically available by the end of Year 11.

<details>
<summary style="background-color:#f0f8ff; padding:8px; border:1px solid #ccc; border-radius:5px; cursor:pointer; font-weight:bold; color:#003366;">
Table 2 — Data Schedule Based on Earliest Applicable Usage (click to expand)
</summary>
<table>
    <tr>
        <th class="group-info">Year Group</th>
        <th class="group-info">Academic Age</th>
        <th class="group-data">Sep / Oct (Autumn Term) Data Sources</th>
        <th class="phase-one">Phase 1: Predict Pre-Spring Outcome</th>
        <th class="group-data">July (Summer Term) Data Sources</th>
        <th class="phase-two">Phase 2: Predict Post-Spring Outcome</th>
    </tr>
    <tr>
        <td>Y7</td>
        <td>11</td>
        <td></td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y7 School Data</td>
        <td></td>
    </tr>
    <tr>
        <td>Y8-Y10</td>
        <td>12-14</td>
        <td></td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + School Data for the Corresponding Year Group</td>
        <td></td>
    </tr>
    <tr>
        <td>Y11</td>
        <td>15</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 16)</td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 16)</td>
        <td>NCCIS September Extract in the Following Year (Age 17)</td>
    </tr>
    <tr>
        <td>Y12</td>
        <td>16</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 16)</td>
        <td>NCCIS March Extract (Age 16)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 16)</td>
        <td>NCCIS September Extract in the Following Year (Age 17)</td>
    </tr>
    <tr>
        <td>Y13</td>
        <td>17</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 17)</td>
        <td>NCCIS March Extract (Age 17)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 17)</td>
        <td>NCCIS September Extract in the Following Year (Age 18)</td>
    </tr>
    <tr>
        <td>Y14+</td>
        <td>18-24</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS September Extract (Age 18+)</td>
        <td>NCCIS March Extract (Age 18+)</td>
        <td>KS2 + Y11 School Data + KS4 + NCCIS March Extract (Age 18+)</td>
        <td>NCCIS September Extract in the Following Year (Age 19+)</td>
    </tr>
</table>
<div class="prediction-note">
    <strong>Prediction outcome timing:</strong><br>
    <strong>Phase 1 - Pre-Spring Outcome</strong> refers to the student's outcome recorded in the March NCCIS extract.<br>
    <strong>Phase 2 - Post-Spring Outcome</strong> refers to the student's outcome recorded in the September NCCIS extract in the following academic year.
</div>
</details>

 > By default, the schedule follows the **Earliest Applicable Usage** method *(append as soon as data is logically available, even if not yet collected in practice)*,  
> meaning that `nccis_append_start_yg` is set to **11** (append KS4 & NCCIS from Year 11 onwards).
> 
> If you wish to switch to the **Actual Availability** method *(append only when the data has been physically collected in reality)*,  
> simply set:
> 
> ```python
> nccis_append_start_yg = 12
> ```
> 
> This will delay appending KS4 & NCCIS until Year 12.

<details>
<summary><strong>Data definitions</strong></summary>


| Item | Definition |
|---|---|
| **Academic Age** | Refers to academic rather than actual age. It increases by one each school year, from Year 7 / Age 11 up to Age 24. |
| **Autumn Term** | September–December |
| **Spring Term** | January–March/April |
| **Summer Term** | April–July |
| **KS2** | Key Stage 2 results collected in Year 6. |
| **KS4** | Key Stage 4 results collected in Year 11 and fully available by Year 12. |
| **School Data** | Census, attendance, and suspension/exclusion records. |
| **NCCIS — Training Data** | Full NCCIS records, including demographics and activity status. |
| **NCCIS — Prediction Target** | NCCIS Code only. |

</details>

<details>
<summary><strong>Data quality treatments</strong></summary>

| Fields | Issue | Treatment |
|---|---|---|
| `ks2_readscore`, `ks2_matscore`, `ks2_gpsscore` | Expected range: `0–999`; value `N` found | Converted `N` to `NaN` |
| `ks2_readscore_nospeccon`, `ks2_matscore_nospeccon`, `ks2_gpsscore_nospeccon` | Value `N` found | Converted `N` to `NaN` |
| `census_date_of_birth` | Invalid non-datetime values found, e.g. `StudID StudID` | Converted invalid values to `NaN` |

</details>

<!-- <details>
<summary><strong>Data type warning</strong></summary> -->

<p style="color:red;">
⚠️ Merging or concatenating DataFrames may change column data types, including converting columns to <code>object</code>. Check the resulting types using <code>df.dtypes</code> after each operation.
</p>

<!-- </details> -->

In [15]:
# Set to False to use the current prepared-dataset filename from the config.
# Set to True to name the file from the available year range, e.g. 2018-2026.parquet.
# Either choice persists the resulting dataset path to .neetml/config.yaml.
name_by_year_range = True

if name_by_year_range:
    print(
        "The final dataset will be named based on the available year range, "
        "e.g., 2018-2026.parquet"
    )
else:
    prepared_path = preprocessor.settings.get_path("prepared")
    print(f"The final dataset will be saved to {prepared_path}")

preprocessor.merge_data(
    overwrite=False,
    save_tmp_outputs=True,
    group_by=False,
    nccis_append_start_yg=11,
    name_by_year_range=name_by_year_range,
)

The final dataset will be named based on the available year range, e.g., 2018-2026.parquet

+------------------------------------+
|  Starting data merging process...  |
+------------------------------------+

2026-07-22 23:11 - INFO - Loading file metadata from default file_metadata_path: /Users/alysia/Research-NEET/data/02_processed/file_metadata.xlsx...
2026-07-22 23:11 - INFO - nccis data has been merged and saved...
2026-07-22 23:11 - INFO - sepGuarantee data has been merged and saved...
2026-07-22 23:11 - INFO - No changes detected and output files exist at /Users/alysia/Research-NEET/data/02_processed/2_merged/2018-2026.parquet. Skipping merge process.
Data merging process completed successfully.


# 10. <a id='toc10_'></a>[Data cleaning: remove duplicate data ](#toc0_)

In [16]:
# preprocessor.clean_data(
#   remove_nan_stud_id=True, 
#   remove_nan_threshold=False,
#   remove_all_nan_columns=True,
#   remove_duplicates_threshold='first',
#   remove_problematic_ids=[471807, 534073]
#   )

<!--  -->